In [10]:
%pip install pyspark==4.0.1 findspark
%pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [12]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col

spark = (
    SparkSession.builder
    .appName("big-data-programming-3")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

In [13]:
df = spark.read.csv(
    "./input/weatherData.csv",
    header=True,
    inferSchema=True,
)

df.show()

+-----------+---------+-----------------------+-----------------------+-----------------------+------------------------+-----------------------------+-----------------------------+------------------------------+---------------------+---------------------+----------------------+-------------+-----------------------+-------------------------+-------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------+-------------------+
|location_id|     date|weather_code (wmo code)|temperature_2m_max (°C)|temperature_2m_min (°C)|temperature_2m_mean (°C)|apparent_temperature_max (°C)|apparent_temperature_min (°C)|apparent_temperature_mean (°C)|daylight_duration (s)|sunshine_duration (s)|precipitation_sum (mm)|rain_sum (mm)|precipitation_hours (h)|wind_speed_10m_max (km/h)|wind_gusts_10m_max (km/h)|wind_direction_10m_dominant (°)|shortwave_radiation_sum (MJ/m²)|et0_fao_evapotranspiration (mm)|            sunrise|           

In [14]:
print(df.count())

142371


In [15]:
df = df.withColumn("year", split(col("date"), "/").getItem(2))
df = df.withColumn("month", split(col("date"), "/").getItem(0))
df = df.withColumn("day", split(col("date"), "/").getItem(1))
df = df.select("location_id", "year", "month", "day", "precipitation_sum (mm)")

df = df.withColumnRenamed("precipitation_sum (mm)", "precipitation")
df.show()

+-----------+----+-----+---+-------------+
|location_id|year|month|day|precipitation|
+-----------+----+-----+---+-------------+
|          0|2010|    1|  1|          0.0|
|          0|2010|    1|  2|          0.1|
|          0|2010|    1|  3|          0.6|
|          0|2010|    1|  4|          0.0|
|          0|2010|    1|  5|          0.0|
|          0|2010|    1|  6|          0.0|
|          0|2010|    1|  7|          0.0|
|          0|2010|    1|  8|          0.8|
|          0|2010|    1|  9|          3.2|
|          0|2010|    1| 10|          0.1|
|          0|2010|    1| 11|          0.7|
|          0|2010|    1| 12|          6.8|
|          0|2010|    1| 13|          4.8|
|          0|2010|    1| 14|          5.7|
|          0|2010|    1| 15|          0.0|
|          0|2010|    1| 16|          3.3|
|          0|2010|    1| 17|          2.7|
|          0|2010|    1| 18|          4.5|
|          0|2010|    1| 19|          4.9|
|          0|2010|    1| 20|          0.0|
+----------

In [16]:
# average by year month
from pyspark.sql.functions import avg

df_avg = df.groupBy("location_id", "year", "month").agg(avg("precipitation").alias("avg_precipitation"))
df_avg = df_avg.orderBy("location_id", "year", "month")
df_avg.show()

+-----------+----+-----+------------------+
|location_id|year|month| avg_precipitation|
+-----------+----+-----+------------------+
|          0|2010|    1|2.9419354838709677|
|          0|2010|   10| 9.087096774193549|
|          0|2010|   11| 16.51666666666667|
|          0|2010|   12|13.103225806451617|
|          0|2010|    2| 2.492857142857143|
|          0|2010|    3| 1.967741935483871|
|          0|2010|    4| 8.693333333333332|
|          0|2010|    5|10.383870967741938|
|          0|2010|    6|12.946666666666665|
|          0|2010|    7| 7.964516129032258|
|          0|2010|    8|               6.5|
|          0|2010|    9| 8.833333333333336|
|          0|2011|    1|5.2451612903225815|
|          0|2011|   10| 9.148387096774192|
|          0|2011|   11| 9.770000000000001|
|          0|2011|   12| 7.425806451612902|
|          0|2011|    2|             4.125|
|          0|2011|    3|3.7967741935483876|
|          0|2011|    4| 9.136666666666667|
|          0|2011|    5|  6.8774

In [19]:
# save to csv
pdf = df_avg.toPandas()  # bring the result to the driver
pdf.to_csv(r"./output/average_precipitation_by_month.csv", index=False)